In [ ]:
import librosa
import soundfile as sf
import numpy as np
import os
from gtts import gTTS
from pydub import AudioSegment
import tempfile

class AudioProcessor:
    def __init__(self):
        self.sample_rate = 22050
        
    def extract_audio(self, video_path):
        try:
            import moviepy.editor as mp
            video = mp.VideoFileClip(video_path)
            
            temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix='.wav')
            temp_audio_path = temp_audio.name
            temp_audio.close()
            
            video.audio.write_audiofile(temp_audio_path, verbose=False, logger=None)
            video.close()
            
            return temp_audio_path
        except Exception as e:
            print(f"Error in audio extraction {e}")
            return None
    
    def process_dubbed_audio(self, dubbed_audio_path, original_audio_path):
        # تحميل الملفات الصوتية
        dubbed_audio, sr1 = librosa.load(dubbed_audio_path, sr=self.sample_rate)
        original_audio, sr2 = librosa.load(original_audio_path, sr=self.sample_rate)
        
        dubbed_audio = self.adjust_duration(dubbed_audio, original_audio)
        
        dubbed_audio = self.match_volume(dubbed_audio, original_audio)
        
        dubbed_audio = self.match_spectral_properties(dubbed_audio, original_audio)
        
        output_path = tempfile.NamedTemporaryFile(delete=False, suffix='.wav').name
        sf.write(output_path, dubbed_audio, self.sample_rate)
        
        return output_path
    
    def adjust_duration(self, dubbed_audio, original_audio):
        target_length = len(original_audio)
        current_length = len(dubbed_audio)
        
        if current_length > target_length:
            return dubbed_audio[:target_length]
        elif current_length < target_length:
            indices = np.linspace(0, current_length - 1, target_length)
            return np.interp(indices, np.arange(current_length), dubbed_audio)
        else:
            return dubbed_audio
    
    def match_volume(self, dubbed_audio, original_audio):
        original_rms = np.sqrt(np.mean(original_audio**2))
        dubbed_rms = np.sqrt(np.mean(dubbed_audio**2))
        
        if dubbed_rms > 0:
            scale_factor = original_rms / dubbed_rms
            dubbed_audio = dubbed_audio * scale_factor
        
        return np.clip(dubbed_audio, -1.0, 1.0)
    
    def match_spectral_properties(self, dubbed_audio, original_audio):
        """مطابقة الخصائص الطيفية"""
        # استخراج ميزات MFCC
        original_mfcc = librosa.feature.mfcc(y=original_audio, sr=self.sample_rate, n_mfcc=13)
        dubbed_mfcc = librosa.feature.mfcc(y=dubbed_audio, sr=self.sample_rate, n_mfcc=13)
        
        # تطبيع الميزات
        original_mfcc = (original_mfcc - np.mean(original_mfcc)) / np.std(original_mfcc)
        dubbed_mfcc = (dubbed_mfcc - np.mean(dubbed_mfcc)) / np.std(dubbed_mfcc)
        
        # تطبيق التحويل الطيفي
        stft_original = librosa.stft(original_audio)
        stft_dubbed = librosa.stft(dubbed_audio)
        
        # مطابقة الطيف
        magnitude_original = np.abs(stft_original)
        phase_dubbed = np.angle(stft_dubbed)
        
        # إعادة بناء الصوت مع الطيف المعدل
        stft_modified = magnitude_original * np.exp(1j * phase_dubbed)
        modified_audio = librosa.istft(stft_modified)
        
        return modified_audio
    
    def text_to_speech(self, text, language='en'):
        try:
            tts = gTTS(text=text, lang=language, slow=False)
            temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
            tts.save(temp_audio.name)
            
            # تحويل إلى WAV
            audio = AudioSegment.from_mp3(temp_audio.name)
            wav_path = temp_audio.name.replace('.mp3', '.wav')
            audio.export(wav_path, format='wav')
            
            os.unlink(temp_audio.name)
            return wav_path
        except Exception as e:
            print(f"Error in Text-to-speech process: {e}")
            return None

c:\Users\hp\miniconda3\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
